# 🥉 Bronze Layer - Purchasing Data Ingestion
## AWS S3 External Storage via Unity Catalog

**Purpose**: Ingest Purchasing schema tables from source to Bronze layer

**Source**: `adventureworks_live_conn_catalog.purchasing.*`
**Target**: `workspace.`workspace-adventureworks-bronze`` (S3)
**Tables**: 5 tables (~13K rows)

**Key Tables**:
- Vendor
- PurchaseOrderHeader
- PurchaseOrderDetail
- ShipMethod

**Transformation**: Add audit columns (ingestion_timestamp, source_system)

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql import Row
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from datetime import datetime
import time

SOURCE_CATALOG = "adventureworks_live_conn_catalog"
SOURCE_SCHEMA = "purchasing"
TARGET_SCHEMA = "workspace.`workspace-adventureworks-bronze`"
SOURCE_SYSTEM = "AdventureWorks_OLTP"

print("=" * 80)
print("🥉 BRONZE LAYER - PURCHASING DATA INGESTION")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: {SOURCE_CATALOG}.{SOURCE_SCHEMA}")
print(f"Target: {TARGET_SCHEMA} (AWS S3 External Storage)")
print("=" * 80)
print()

ingestion_results = []

def ingest_table_to_bronze(source_table_name):
    source_full = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{source_table_name}"
    target_full = f"{TARGET_SCHEMA}.{source_table_name}"
    
    result = {"source_table": source_table_name, "status": "failed", "row_count": 0, "duration_seconds": 0, "error": None}
    
    try:
        start_time = time.time()
        print(f"\n⏳ Ingesting: {source_table_name}...", end=" ")
        
        df = spark.table(source_full)
        df_bronze = df.withColumn("ingestion_timestamp", current_timestamp()).withColumn("source_system", lit(SOURCE_SYSTEM))
        
        df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_full)
        
        row_count = df_bronze.count()
        duration = time.time() - start_time
        result.update({"status": "success", "row_count": row_count, "duration_seconds": round(duration, 2)})
        print(f"✅ {row_count:,} rows in {duration:.2f}s")
    except Exception as e:
        result.update({"error": str(e), "duration_seconds": round(time.time() - start_time, 2)})
        print(f"❌ Failed: {str(e)}")
    
    return result

# Discover and ingest
print("🔍 Discovering Purchasing tables...")
tables = spark.sql(f"SHOW TABLES IN {SOURCE_CATALOG}.{SOURCE_SCHEMA}").collect()
table_names = [row.tableName for row in tables]
print(f"✅ Found {len(table_names)} tables\n")

print("=" * 80)
print("🚀 STARTING INGESTION")
print("=" * 80)

overall_start = time.time()
for table_name in table_names:
    ingestion_results.append(ingest_table_to_bronze(table_name))

overall_duration = time.time() - overall_start

print("\n" + "=" * 80)
print("🏁 INGESTION COMPLETE")
print("=" * 80)
print(f"Total Duration: {overall_duration:.2f}s\n")

success_count = sum(1 for r in ingestion_results if r["status"] == "success")
failed_count = sum(1 for r in ingestion_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in ingestion_results)

print("📊 SUMMARY")
print(f"  Tables: {len(ingestion_results)} | Success: {success_count} | Failed: {failed_count}")
print(f"  Total Rows: {total_rows:,}\n")

# Create summary DataFrame with explicit schema
schema = StructType([
    StructField("source_table", StringType(), False),
    StructField("status", StringType(), False),
    StructField("row_count", IntegerType(), False),
    StructField("duration_seconds", DoubleType(), False),
    StructField("error", StringType(), True)
])

summary_data = [
    Row(
        source_table=r["source_table"],
        status=r["status"],
        row_count=r["row_count"],
        duration_seconds=r["duration_seconds"],
        error=r["error"]
    )
    for r in ingestion_results
]

summary_df = spark.createDataFrame(summary_data, schema=schema)
display(summary_df.orderBy("status", "source_table"))

print("=" * 80)
if failed_count == 0:
    print("✅ BRONZE PURCHASING INGESTION COMPLETED SUCCESSFULLY")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ COMPLETED WITH ERRORS")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} failed")

In [0]:
# Show detailed errors from the ingestion
failed_tables = [r for r in ingestion_results if r["status"] == "failed"]

print(f"\n{'='*80}")
print(f"FAILED TABLES: {len(failed_tables)}")
print(f"{'='*80}\n")

for failed in failed_tables:
    print(f"\n❌ Table: {failed['source_table']}")
    print(f"   Error: {failed['error']}")
    print(f"   Duration: {failed['duration_seconds']}s")
    print("-" * 80)